# Lezione 10 · Pratica: apprendimento non supervisionato e rappresentazioni

In questa pratica applichi quanto visto nella Lezione 10 sull'apprendimento non supervisionato e sulle rappresentazioni. Lavori con un autocodificatore (*autoencoder*), la rete che comprime un'immagine in un vettore latente e la ricostruisce a partire da esso, e usi l'errore di ricostruzione per individuare esempi anomali.

## Cosa farai in questa pratica

- Completi l'architettura di un autocodificatore con strati lineari in PyTorch e lo addestri sul dataset di cifre scritte a mano MNIST (Modified National Institute of Standards and Technology)
- Usi l'errore di ricostruzione di un autocodificatore già addestrato per individuare le immagini di test ricostruite peggio, un indizio del fatto che siano anomale o difficili

## Prerequisito

Prima di iniziare, assicurati di aver seguito per intero la Lezione 10: devi capire cos'è un codificatore, un decodificatore, il collo di bottiglia latente e perché l'errore di ricostruzione funziona come segnale di anomalia.

Ogni esercizio ha lo scheletro del codice già scritto. Completa solo le parti contrassegnate con `# TODO`. Quando hai finito, apri la soluzione per confrontare il tuo codice, ma prova prima a risolverlo da solo.

Questo notebook è pensato per Google Colab e usa solo librerie già installate: PyTorch, torchvision e matplotlib. Se Colab ti assegna una GPU (unità di elaborazione grafica), l'addestramento è più veloce, ma il notebook funziona anche solo con CPU (unità centrale di elaborazione).

## Esercizio 1 — Individua cifre anomale con un autocodificatore

### Il caso

Lavori in un team che digitalizza moduli compilati a mano e vuole individuare automaticamente le cifre scansionate male o direttamente illeggibili, prima che arrivino a un sistema di riconoscimento ottico dei caratteri (OCR). Non hai etichette di "normale" o "anomalo": hai solo un file con migliaia di cifre scansionate. La soluzione non supervisionata è addestrare un autocodificatore sulle cifre normali; quelle che la rete ricostruisce peggio sono i candidati più probabili a essere anomali o mal scansionati.

In questo esercizio completi l'architettura dell'autocodificatore e lo addestri su MNIST. Nell'esercizio 2 userai questo modello addestrato per individuare i casi peggiori.

### Cosa devi fare

Completa la classe `Autoencoder`:

- Il **codificatore** (`self.codificador`) è un `nn.Sequential` che comprime progressivamente il vettore di ingresso: da `dim_entrada` a 256, da 256 a 128 e da 128 a `dim_latente`, con un'attivazione `nn.ReLU()` tra ogni strato lineare
- Il **decodificatore** (`self.decodificador`) percorre il cammino inverso: da `dim_latente` a 128, da 128 a 256 e da 256 a `dim_entrada`, anch'esso con `nn.ReLU()` tra gli strati, e una `nn.Sigmoid()` finale, perché i pixel di MNIST sono normalizzati tra zero e uno
- Il metodo `forward` fa passare `x` attraverso il codificatore per ottenere il vettore latente `z`, e `z` attraverso il decodificatore per produrre la ricostruzione

Il resto dello scheletro — caricamento dei dati, funzione di perdita, ottimizzatore e ciclo di addestramento — è già scritto. Non modificarlo: completalo solo con la tua classe.

**Suggerimento**: osserva l'esempio di codice commentato che hai visto nel capitolo del manuale. L'architettura richiesta da questo esercizio è esattamente quella.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Usa GPU si Colab te la ha asignado, si no, usa CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

# --- Datos ---
# MNIST: dígitos de 28x28 en escala de grises, valores normalizados entre 0 y 1
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

DIM_ENTRADA = 28 * 28   # cada imagen se aplana a un vector de 784 valores
DIM_LATENTE = 32        # tamaño del cuello de botella: el "resumen" de cada imagen


# --- Definición del autocodificador ---
class Autoencoder(nn.Module):
    def __init__(self, dim_entrada=DIM_ENTRADA, dim_latente=DIM_LATENTE):
        super().__init__()
        # TODO: define el codificador como nn.Sequential.
        # Comprime dim_entrada -> 256 -> 128 -> dim_latente, con nn.ReLU()
        # entre cada capa nn.Linear.
        self.codificador = ...

        # TODO: define el decodificador como nn.Sequential.
        # Expande dim_latente -> 128 -> 256 -> dim_entrada, con nn.ReLU()
        # entre capas lineales y una nn.Sigmoid() final (píxeles en [0, 1]).
        self.decodificador = ...

    def forward(self, x):
        # TODO: calcula z pasando x por el codificador, y x_reconstruida
        # pasando z por el decodificador.
        z = ...
        x_reconstruida = ...
        return x_reconstruida, z

In [ ]:
modelo = Autoencoder().to(device)
print(modelo)

# El error de reconstrucción binario funciona bien con píxeles en [0, 1] y salida sigmoide
criterio = nn.BCELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)


def entrenar_una_epoca(modelo, loader, criterio, optimizador):
    modelo.train()
    perdida_total = 0.0
    for imagenes, _ in loader:  # las etiquetas no se usan: no hace falta supervisión
        imagenes = imagenes.view(imagenes.size(0), -1).to(device)  # aplana a vector

        optimizador.zero_grad()
        reconstruidas, _ = modelo(imagenes)
        perdida = criterio(reconstruidas, imagenes)
        perdida.backward()
        optimizador.step()

        perdida_total += perdida.item() * imagenes.size(0)
    return perdida_total / len(loader.dataset)


num_epocas = 10
for epoca in range(1, num_epocas + 1):
    perdida_train = entrenar_una_epoca(modelo, train_loader, criterio, optimizador)
    print(f"Época {epoca}/{num_epocas} | pérdida de reconstrucción: {perdida_train:.4f}")

In [ ]:
# Comprobación visual rápida: compara diez originales con sus reconstrucciones
modelo.eval()
imagenes_prueba, _ = next(iter(test_loader))
imagenes_prueba = imagenes_prueba[:10]

with torch.no_grad():
    entrada_plana = imagenes_prueba.view(imagenes_prueba.size(0), -1).to(device)
    reconstruidas, _ = modelo(entrada_plana)
    reconstruidas = reconstruidas.view(-1, 28, 28).cpu()

fig, ejes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    ejes[0, i].imshow(imagenes_prueba[i].squeeze(), cmap="gray")
    ejes[0, i].axis("off")
    ejes[1, i].imshow(reconstruidas[i], cmap="gray")
    ejes[1, i].axis("off")

ejes[0, 0].set_title("Original", loc="left", fontsize=10)
ejes[1, 0].set_title("Reconstruida", loc="left", fontsize=10)
plt.tight_layout()
plt.show()

<details>
<summary>💡 Vedi soluzione</summary>

```python
class Autoencoder(nn.Module):
    def __init__(self, dim_entrada=DIM_ENTRADA, dim_latente=DIM_LATENTE):
        super().__init__()
        # Codificatore: comprime progressivamente fino al collo di bottiglia
        self.codificador = nn.Sequential(
            nn.Linear(dim_entrada, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, dim_latente),
        )
        # Decodificatore: espande di nuovo fino alla dimensione originale
        # Sigmoid finale perché i pixel di MNIST sono normalizzati in [0, 1]
        self.decodificador = nn.Sequential(
            nn.Linear(dim_latente, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, dim_entrada),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.codificador(x)
        x_reconstruida = self.decodificador(z)
        return x_reconstruida, z
```
</details>

## Esercizio 2 — Trova le dieci immagini ricostruite peggio

### Il caso

Hai già l'autocodificatore addestrato dell'esercizio 1. Ora arriva la parte che interessa davvero al team: trasformare l'errore di ricostruzione in una lista concreta di sospetti. L'idea è semplice: più alto è l'errore di ricostruzione, meno la cifra assomiglia a ciò che la rete ha visto durante l'addestramento, e più è probabile che sia anomala o mal scansionata.

### Cosa devi fare

Completa la funzione `calcular_errores_reconstruccion`, che scorre il set di test per lotti e calcola l'errore quadratico medio (MSE) di ogni immagine, non del lotto intero.

- Dentro il ciclo, ottieni la ricostruzione del lotto con il modello già addestrato
- Calcola l'errore per immagine. Eleva al quadrato la differenza tra l'ingresso appiattito e la ricostruzione, e riduci con `.mean(dim=1)`: così ottieni un unico valore per immagine, non un valore per lotto
- Dopo aver calcolato `errores_test` per tutto il set di test, usa `torch.topk` per ottenere gli indici delle dieci immagini con errore maggiore

**Suggerimento**: `nn.functional.mse_loss` con `reduction="none"` serve anch'esso a ottenere l'errore senza ridurlo, ma qui è più diretto farlo a mano con `** 2` e `.mean(dim=1)`.

In [ ]:
def calcular_errores_reconstruccion(modelo, dataset, device):
    """
    Calcula el error de reconstrucción (MSE) de cada imagen del dataset.

    Devuelve:
        errores: tensor 1D de forma (N,), un valor de error por imagen
        imagenes: tensor de forma (N, 28, 28) con las imágenes originales
    """
    modelo.eval()
    loader = DataLoader(dataset, batch_size=256, shuffle=False)

    errores = []
    imagenes_todas = []

    with torch.no_grad():
        for imagenes, _ in loader:
            entrada_plana = imagenes.view(imagenes.size(0), -1).to(device)

            # TODO: obtén la reconstrucción del modelo para este lote
            reconstruidas, _ = ...

            # TODO: calcula el error cuadrático medio POR IMAGEN.
            # Reduce sobre la dimensión de píxeles (dim=1), no sobre el lote:
            # el resultado debe tener forma (tamaño_del_lote,), un valor por imagen.
            error_por_imagen = ...

            errores.append(error_por_imagen.cpu())
            imagenes_todas.append(imagenes.view(-1, 28, 28))

    errores = torch.cat(errores)
    imagenes_todas = torch.cat(imagenes_todas)
    return errores, imagenes_todas


errores_test, imagenes_test = calcular_errores_reconstruccion(modelo, test_dataset, device)
print(f"Error medio de reconstrucción en test: {errores_test.mean():.4f}")

In [ ]:
N_PEORES = 10

# TODO: obtén los índices de las N_PEORES imágenes con mayor error de
# reconstrucción a partir de 'errores_test'. Pista: torch.topk(tensor, k)
# devuelve (valores, índices) ordenados de mayor a menor.
indices_peores = ...

fig, ejes = plt.subplots(1, N_PEORES, figsize=(15, 2))
for i, idx in enumerate(indices_peores):
    ejes[i].imshow(imagenes_test[idx], cmap="gray")
    ejes[i].set_title(f"{errores_test[idx]:.3f}", fontsize=8)
    ejes[i].axis("off")

plt.suptitle("Las diez imágenes con mayor error de reconstrucción (candidatas a anómalas)")
plt.tight_layout()
plt.show()

<details>
<summary>💡 Vedi soluzione</summary>

```python
def calcular_errores_reconstruccion(modelo, dataset, device):
    modelo.eval()
    loader = DataLoader(dataset, batch_size=256, shuffle=False)

    errores = []
    imagenes_todas = []

    with torch.no_grad():
        for imagenes, _ in loader:
            entrada_plana = imagenes.view(imagenes.size(0), -1).to(device)

            reconstruidas, _ = modelo(entrada_plana)

            # Errore quadratico medio per immagine: si riduce sui pixel (dim=1)
            error_por_imagen = ((entrada_plana - reconstruidas) ** 2).mean(dim=1)

            errores.append(error_por_imagen.cpu())
            imagenes_todas.append(imagenes.view(-1, 28, 28))

    errores = torch.cat(errores)
    imagenes_todas = torch.cat(imagenes_todas)
    return errores, imagenes_todas


errores_test, imagenes_test = calcular_errores_reconstruccion(modelo, test_dataset, device)
print(f"Errore medio di ricostruzione sul test: {errores_test.mean():.4f}")

N_PEORES = 10
valores_peores, indices_peores = torch.topk(errores_test, N_PEORES)

fig, ejes = plt.subplots(1, N_PEORES, figsize=(15, 2))
for i, idx in enumerate(indices_peores):
    ejes[i].imshow(imagenes_test[idx], cmap="gray")
    ejes[i].set_title(f"{errores_test[idx]:.3f}", fontsize=8)
    ejes[i].axis("off")

plt.suptitle("Le dieci immagini con maggior errore di ricostruzione (candidate anomale)")
plt.tight_layout()
plt.show()
```
</details>

## Chiusura

Hai appena trasformato una rete che impara solo a copiare il proprio ingresso in un rilevatore di anomalie senza una sola etichetta. Lo stesso schema — addestrare su dati normali e usare l'errore di ricostruzione come segnale — si applica al rilevamento di frodi, alla manutenzione predittiva o al controllo qualità industriale, non solo alle cifre scritte a mano.

Prima di passare al capitolo successivo, prova a cambiare `DIM_LATENTE` e osserva come si sposta l'elenco delle dieci immagini ricostruite peggio: un collo di bottiglia più stretto tende a segnalare come anomale anche cifre ragionevolmente normali, perché il modello perde capacità di rappresentarle bene.

**Salva il tuo notebook completato e condividilo con il tuo team: vedere quali cifre ogni modello segnala come "più anomala" è il modo migliore per intuire cosa impara davvero lo spazio latente.**